# Final seasonal-Poisson six-week outbreak forecast

This notebook applies the final all-data seasonal Poisson parameter vector at the most recent complete London observation in the UKHSA export. It excludes weeks marked as within the reporting-delay period, estimates the hidden state from the four most recent complete weekly counts, simulates 1,000 stochastic six-week paths, and reports the probability that at least one weekly count is strictly greater than 15 cases.

This is the final-origin forecast, not a model-validation experiment. Use `London_Strict_Forecast_Validation.ipynb` for held-out performance and `London_Seasonal_Complete_Rolling_Audit.ipynb` for the complete historical visual audit.

The forecast figure includes the observed forecast-origin count as week 0. Week 0 is contextual only; outbreak probability is calculated from simulated forecast weeks 1--6.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'outbreak_probability_model': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from outbreak_probability_model.data_loader import (
    DEFAULT_UKHSA_CASES, load_london_observed,
)
from outbreak_probability_model.model import load_default_inputs
from outbreak_probability_model.london_calibration import (
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS,
    HistoryConditioningConfig, forecast_london_age_groups,
    load_london_fitted_parameters,
)

UKHSA_CASES_PATH = DEFAULT_UKHSA_CASES
OUTPUT = ROOT / 'experiments' / 'measles' / 'London' / 'seasonal_poisson_outbreak_probability'
OUTPUT.mkdir(parents=True, exist_ok=True)
N_SIMULATIONS = 1000
HORIZON_WEEKS = 6
OUTBREAK_THRESHOLD = 15.0
RANDOM_SEED = 20260826
history_conditioning = HistoryConditioningConfig(
    history_weeks=4, transmission_multiplier_bounds=(0.8, 1.25),
    regularization_strength=1.0, origin_observation_weight=4.0, maxiter=80,
)

## 1. Load and record the final fitted model

In [ ]:
if not DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS.exists():
    raise FileNotFoundError(
        'Run London_Seasonal_Poisson_Fit.ipynb first; missing ' +
        str(DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS)
    )
if not UKHSA_CASES_PATH.exists():
    raise FileNotFoundError('Missing UKHSA export: ' + str(UKHSA_CASES_PATH))
cases = load_london_observed(
    UKHSA_CASES_PATH, exclude_reporting_delay=True
)[["date", "observed_cases"]]
if len(cases) < history_conditioning.history_weeks:
    raise ValueError('Not enough complete weekly observations for history conditioning.')
inputs = load_default_inputs()
fitted_parameters, fitted_vector = load_london_fitted_parameters(
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS
)
settings = pd.DataFrame([{
    'forecast_origin': cases.date.iloc[-1],
    'origin_reported_cases': cases.observed_cases.iloc[-1],
    'case_input_path': str(UKHSA_CASES_PATH),
    'reporting_delay_weeks_excluded': True,
    'simulations': N_SIMULATIONS, 'horizon_weeks': HORIZON_WEEKS,
    'outbreak_threshold_strictly_greater_than': OUTBREAK_THRESHOLD,
    'history_weeks': history_conditioning.history_weeks,
    'noise_scale': fitted_parameters.noise_scale,
    'sh_noise_multiplier': fitted_parameters.sh_noise_multiplier,
    'seasonal_amplitude': fitted_parameters.seasonal_amplitude,
    'seasonal_peak_week': fitted_parameters.seasonal_peak_week,
    'random_seed': RANDOM_SEED,
}])
settings.to_csv(OUTPUT / '00_final_forecast_settings.csv', index=False)
cases.tail(history_conditioning.history_weeks).to_csv(
    OUTPUT / '00a_history_conditioning_weeks.csv', index=False
)
pd.DataFrame([fitted_vector]).to_csv(OUTPUT / '01_fitted_parameter_vector.csv', index=False)
display(settings)
display(pd.DataFrame({'parameter': fitted_vector.keys(), 'value': fitted_vector.values()}))

## 2. Run the stochastic forecast

In [ ]:
forecast = forecast_london_age_groups(
    cases=cases, inputs=inputs,
    fitted_parameters_path=DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS,
    outbreak_threshold=OUTBREAK_THRESHOLD, horizon_weeks=HORIZON_WEEKS,
    n_simulations=N_SIMULATIONS, random_seed=RANDOM_SEED,
    warmup_weeks=0, sample_weekly_counts=True,
    history_conditioning=history_conditioning,
)
summary = pd.DataFrame([{
    'forecast_origin': cases.date.iloc[-1],
    'forecast_start': forecast.forecast_start,
    'outbreak_probability': forecast.all_age_probability,
    'monte_carlo_95_low': forecast.all_age_mc95_low,
    'monte_carlo_95_high': forecast.all_age_mc95_high,
    'any_age_group_probability': forecast.any_age_group_probability,
    'threshold': OUTBREAK_THRESHOLD, 'simulations': N_SIMULATIONS,
}])
summary.to_csv(OUTPUT / '02_outbreak_probability.csv', index=False)
forecast.weekly_summary.to_csv(OUTPUT / '03_age_weekly_summary.csv', index=False)
forecast.all_age_trajectories.to_csv(OUTPUT / '04_all_age_paths.csv', index=False)
forecast.trajectories.to_csv(OUTPUT / '05_age_paths.csv', index=False)
forecast.age_probabilities.to_csv(OUTPUT / '06_age_probabilities.csv', index=False)
forecast.history_conditioning_summary.to_csv(
    OUTPUT / '07_history_conditioning.csv', index=False
)
display(summary.style.format({
    'outbreak_probability': '{:.1%}', 'monte_carlo_95_low': '{:.1%}',
    'monte_carlo_95_high': '{:.1%}', 'any_age_group_probability': '{:.1%}',
}))
display(forecast.history_conditioning_summary)

## 3. Stochastic paths and age-specific contributions

The London outbreak probability is calculated after summing cases across all age groups. An age-specific probability instead asks whether one age group alone crosses the same 15-case weekly threshold. These probabilities are therefore not components that should sum to the all-age probability. The figures below show the all-age event and the underlying age-specific paths separately.

In [ ]:
wide = forecast.all_age_trajectories.pivot(
    index='week', columns='simulation', values='weekly_cases'
).sort_index()
q10 = wide.quantile(.10, axis=1)
q50 = wide.quantile(.50, axis=1)
q90 = wide.quantile(.90, axis=1)

# Week 0 is the observed forecast-origin count. It is shown for context but
# is not included when calculating the six-week outbreak probability.
origin_cases = float(cases.observed_cases.iloc[-1])
plot_weeks = np.arange(0, HORIZON_WEEKS + 1)
q10_plot = np.r_[origin_cases, q10.to_numpy(float)]
q50_plot = np.r_[origin_cases, q50.to_numpy(float)]
q90_plot = np.r_[origin_cases, q90.to_numpy(float)]

N_DISPLAY_PATHS = min(20, wide.shape[1])
display_positions = np.linspace(0, wide.shape[1] - 1, N_DISPLAY_PATHS, dtype=int)
display_simulations = wide.columns[display_positions]

# All-age forecast. Only 20 paths are drawn so individual trajectories remain visible;
# all simulations still contribute to the median, interval and probability.
fig, ax = plt.subplots(figsize=(8.2, 5.0))
for path_number, simulation in enumerate(display_simulations):
    path = np.r_[origin_cases, wide[simulation].to_numpy(float)]
    ax.plot(
        plot_weeks, path, color='#4f9dcc', alpha=.38, lw=1.0,
        label='20 displayed stochastic paths' if path_number == 0 else None,
    )
ax.fill_between(
    plot_weeks, q10_plot, q90_plot,
    color='#8ecae6', alpha=.22, label='forecast p10--p90',
)
ax.plot(plot_weeks, q50_plot, color='#023047', lw=2.5, label='forecast median')
ax.scatter([0], [origin_cases], color='black', s=46, zorder=5,
           label='observed week 0')
ax.axhline(OUTBREAK_THRESHOLD, color='#d62828', lw=1.5, ls='--',
           label=f'all-age threshold (> {OUTBREAK_THRESHOLD:g})')
ax.set(
    title=(f'All-age London forecast: '
           f'P(any forecast week > {OUTBREAK_THRESHOLD:g}) = '
           f'{forecast.all_age_probability:.0%}'),
    xlabel='Week (0 = observed forecast origin)',
    ylabel='Reported cases per week',
)
ax.set_xticks(plot_weeks)
ax.grid(alpha=.25)
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT / '08_final_forecast_summary.png', dpi=180,
            bbox_inches='tight')
plt.show()

# Age-specific contributions. Week 0 is allocated using the latest available
# age distribution; it is not a directly observed weekly age-specific count.
age_weights = forecast.age_allocation.set_index('age_group')['allocation_weight']
age_probabilities = forecast.age_probabilities.set_index('age_group')['outbreak_probability']
age_groups = list(inputs.age_groups)
fig, axes = plt.subplots(2, 4, figsize=(15, 7.4), sharex=True)
axes = axes.ravel()
for panel_number, (ax, age_group) in enumerate(zip(axes, age_groups)):
    age_wide = (
        forecast.trajectories.loc[lambda d: d.age_group.eq(age_group)]
        .pivot(index='week', columns='simulation', values='weekly_cases')
        .sort_index()
    )
    age_origin = origin_cases * float(age_weights.loc[age_group])
    age_q10 = np.r_[age_origin, age_wide.quantile(.10, axis=1).to_numpy(float)]
    age_q50 = np.r_[age_origin, age_wide.quantile(.50, axis=1).to_numpy(float)]
    age_q90 = np.r_[age_origin, age_wide.quantile(.90, axis=1).to_numpy(float)]
    for path_number, simulation in enumerate(display_simulations):
        path = np.r_[age_origin, age_wide[simulation].to_numpy(float)]
        ax.plot(
            plot_weeks, path, color='#f28e5b', alpha=.34, lw=.9,
            label='20 displayed paths'
            if panel_number == 0 and path_number == 0 else None,
        )
    ax.fill_between(
        plot_weeks, age_q10, age_q90, color='#f6bd60', alpha=.22,
        label='p10--p90 (all paths)' if panel_number == 0 else None,
    )
    ax.plot(
        plot_weeks, age_q50, color='#c44536', lw=2.1,
        label='median (all paths)' if panel_number == 0 else None,
    )
    ax.scatter(
        [0], [age_origin], color='black', s=24, zorder=5,
        label='allocated week 0' if panel_number == 0 else None,
    )
    probability = float(age_probabilities.loc[age_group])
    age_label = age_group.replace('_', '-')
    ax.set_title(
        f'{age_label}\n'
        f'P(this group alone > {OUTBREAK_THRESHOLD:g}) = {probability:.1%}',
        fontsize=10,
    )
    ax.set_xticks(plot_weeks)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=.22)
key_ax = axes[-1]
key_ax.axis('off')
handles, labels = axes[0].get_legend_handles_labels()
key_ax.legend(handles, labels, loc='center', frameon=False,
              title='Age-panel key', fontsize=9)
key_ax.text(
    .5, .18,
    f'Each probability uses the same\nweekly threshold > {OUTBREAK_THRESHOLD:g}.',
    ha='center', va='center', fontsize=9, transform=key_ax.transAxes,
)
for ax in axes[4:7]:
    ax.set_xlabel('Forecast week')
axes[0].set_ylabel('Reported cases per week')
axes[4].set_ylabel('Reported cases per week')
fig.suptitle(
    'Age-specific stochastic contributions to the all-age London forecast\n'
    '(20 displayed paths; summaries use all simulations)',
    y=1.01, fontsize=13,
)
fig.text(
    .5, .005,
    'Week 0 is allocated from the observed all-age count using the latest '
    'available age distribution; it is not a separately observed age count.',
    ha='center', fontsize=9, color='#444444',
)
fig.tight_layout(rect=(0, .035, 1, .97))
fig.savefig(OUTPUT / '09_final_forecast_age_specific_paths.png', dpi=180,
            bbox_inches='tight')
plt.show()


## Reporting

Report the forecast origin, six-week horizon, strict threshold definition, number of paths, probability and Monte Carlo interval together. The Monte Carlo interval measures numerical precision from 1,000 paths; it is not a confidence interval covering all structural or parameter uncertainty.